In [1]:
import pandas as pd
import pandas as pd
import os
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
nltk.download("stopwords")
df = pd.read_csv("../data/raw_reviews.csv")
print("Before cleaning:", df.shape)
print("Missing values before cleaning:")
print(df.isnull().sum())

Before cleaning: (654, 12)
Missing values before cleaning:
review_id              0
item_name              0
review_title         360
review_text            0
rating                60
review_date          300
helpful_votes          0
total_votes            0
helpfulness_score      0
helpful_label          0
source                 0
review_url             0
dtype: int64


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\samam\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
df = df.replace("", pd.NA)
df = df.replace(r"^\s*$", pd.NA, regex=True)
df = df.drop_duplicates()
df = df.drop_duplicates(subset=["review_text"])
df = df.dropna(subset=["review_text"])
df = df[df["review_text"].str.strip() != ""]
df = df.dropna(subset=["review_text", "rating"])

In [3]:
# check lw al titles a8lbha fdya
null_percentage = df.isnull().mean() * 100
print("\nNull percentage:")
print(null_percentage)
if "review_title" in df.columns and null_percentage["review_title"] > 50:
    df = df.drop(columns=["review_title"])


Null percentage:
review_id             0.000000
item_name             0.000000
review_title         50.590219
review_text           0.000000
rating                0.000000
review_date          50.590219
helpful_votes         0.000000
total_votes           0.000000
helpfulness_score     0.000000
helpful_label         0.000000
source                0.000000
review_url            0.000000
dtype: float64


In [4]:
stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()

In [5]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"posted:\s*\d{1,2}\s+[a-zA-Z]+,?\s*\d{4}", " ", text)
    text = re.sub(r"posted:\s*[a-zA-Z]+\s+\d{1,2},?\s*\d{4}", " ", text)
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"[^\x00-\x7F]+", " ", text)
    text = re.sub(r"\d+", " ", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    tokens = text.split()
    tokens = [w for w in tokens if w not in stop_words]
    tokens = [w for w in tokens if len(w) > 2]
    tokens = [w for w in tokens if w not in ["posted", "post"]]

    stemmed_tokens = [stemmer.stem(w) for w in tokens]
    processed_text = " ".join(stemmed_tokens)

    return text, tokens, processed_text

In [6]:
df[["clean_text", "tokens", "processed_text"]] = df["review_text"].apply(
    lambda x: pd.Series(clean_text(x))
)

In [7]:
# remove empty processed reviews
df = df[df["processed_text"].str.strip() != ""]
df = df[~df["processed_text"].str.contains("posted", na=False)]

In [8]:
df["text_length"] = df["processed_text"].str.len()
df["word_count"] = df["processed_text"].apply(lambda x: len(x.split()))

In [9]:
df = df[df["word_count"] >= 3]

In [10]:
df["rating"] = pd.to_numeric(df["rating"], errors="coerce")
df["rating"] = df["rating"].fillna(df["rating"].median())
df["helpful_votes"] = pd.to_numeric(df["helpful_votes"], errors="coerce").fillna(0).astype(int)
df["total_votes"] = pd.to_numeric(df["total_votes"], errors="coerce").fillna(1).astype(int)
df["total_votes"] = df["total_votes"].replace(0, 1)
df["helpfulness_score"] = df["helpful_votes"] / df["total_votes"]

In [11]:
if "helpful_label" in df.columns:
    df["helpful_label"] = pd.to_numeric(df["helpful_label"], errors="coerce").fillna(0).astype(int)
else:
    df["helpful_label"] = df["helpfulness_score"].apply(lambda x: 1 if x >= 0.5 else 0)

In [12]:
if "review_title" in df.columns:
    df["review_title"] = df["review_title"].fillna("No title")

In [13]:
df["review_date"] = df["review_date"].fillna("Unknown")
df["source"] = df["source"].fillna("Unknown")
df["item_name"] = df["item_name"].fillna("Unknown")

In [14]:
df = df.reset_index(drop=True)

In [15]:
data_dir = "../data"
os.makedirs(data_dir, exist_ok=True)
output_path = os.path.join(data_dir, "clean_reviews.csv")
df.to_csv(output_path, index=False, encoding="utf-8-sig")
df.to_json("../data/clean_reviews.json", orient="records", indent=4, force_ascii=False)

print("Saved to:", output_path)
print("After preprocessing:", df.shape)
print("\nMissing values:")
print(df.isnull().sum())
print("\nDuplicate review_text:", df["review_text"].duplicated().sum())
print("\nHelpful label distribution:")
print(df["helpful_label"].value_counts())
print("\nSaved to:", output_path)
print(df.head())

Saved to: ../data\clean_reviews.csv
After preprocessing: (555, 16)

Missing values:
review_id            0
item_name            0
review_text          0
rating               0
review_date          0
helpful_votes        0
total_votes          0
helpfulness_score    0
helpful_label        0
source               0
review_url           0
clean_text           0
tokens               0
processed_text       0
text_length          0
word_count           0
dtype: int64

Duplicate review_text: 0

Helpful label distribution:
helpful_label
0    293
1    262
Name: count, dtype: int64

Saved to: ../data\clean_reviews.csv
   review_id item_name                                        review_text  \
0          1    Dota 2  Posted: 23 April, 2022\nIf you told me when I ...   
1          2    Dota 2  Posted: 16 April, 2024\nThis community is so n...   
2          3    Dota 2  Posted: 14 November, 2023\nAfter 5000 hours I ...   
3          4    Dota 2  Posted: 13 October, 2025\nThere is nothing tid...   
